In [0]:
# Azure ADLS Gen2 Configuration
storage_account_name = "<STORAGE_ACCOUNT_NAME>"
client_id = "<CLIENT_ID>"
tenant_id = "<TENANT_ID>"
client_secret = "<CLIENT_SECRET>"

# Configure OAuth authentication
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Storage account key (fallback)
spark.conf.set(
    "fs.azure.account.key.<STORAGE_ACCOUNT_NAME>.dfs.core.windows.net",
    "<STORAGE_ACCOUNT_KEY>")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Repair Silver users Delta table (fix any metadata inconsistencies)
silver_base_path = "abfss://silver@ecomadlskyle.dfs.core.windows.net/"
spark.sql(f"FSCK REPAIR TABLE delta.`{silver_base_path}users`")

DataFrame[dataFilePath: string, dataFileMissing: boolean, deletionVectorPath: string, deletionVectorFileMissing: boolean]

In [0]:
spark = SparkSession.builder.appName("GoldLayerCreation").getOrCreate()

# Read all Silver layer tables
silver_base_path = "abfss://silver@ecomadlskyle.dfs.core.windows.net/"

silver_users = spark.read.format("delta").load(f"{silver_base_path}users")
silver_sellers = spark.read.format("delta").load(f"{silver_base_path}sellers")
silver_buyers = spark.read.format("delta").load(f"{silver_base_path}buyers")
silver_countries = spark.read.format("delta").load(f"{silver_base_path}countries")

In [0]:
# =============================================================================
# GOLD LAYER: JOIN OPERATIONS
# =============================================================================
# Key design decisions:
# 1. Pre-aggregate sellers by country BEFORE join (sellers table has male/female
#    rows per country - without aggregation, each user would be duplicated)
# 2. Use LEFT joins (users is the primary table; others enrich it)
# 3. Use deduplicated join key via col("country")

# Pre-aggregate sellers by country to prevent row explosion
silver_sellers_agg = silver_sellers.groupBy("country").agg(
    sum("nbsellers").alias("nbsellers"),
    round(avg("meanproductssold"), 2).alias("meanproductssold"),
    round(avg("meanproductslisted"), 2).alias("meanproductslisted")
)

# Join: users LEFT JOIN countries, buyers, sellers (aggregated)
comprehensive_user_table = silver_users \
    .join(silver_countries, ["country"], "left") \
    .join(silver_buyers, ["country"], "left") \
    .join(silver_sellers_agg, ["country"], "left")

# Select and alias columns for the final Gold table
comprehensive_user_table = comprehensive_user_table.select(
    silver_users["identifierHash"].alias("IdentifierHash"),
    col("country").alias("Country"),
    
    # User metrics
    silver_users["productsSold"].alias("Users_productsSold"),
    silver_users["productsWished"].alias("Users_productsWished"),
    silver_users["hasanyapp"].alias("Users_hasanyapp"),
    silver_users["socialnbfollowers"].alias("Users_socialnbfollowers"),
    
    # Country-level seller stats
    silver_countries["sellers"].alias("Countries_Sellers"),
    silver_countries["topsellers"].alias("Countries_TopSellers"),
    silver_countries["femalesellers"].alias("Countries_FemaleSellers"),
    silver_countries["malesellers"].alias("Countries_MaleSellers"),
    silver_countries["topfemalesellers"].alias("Countries_TopFemaleSellers"),
    silver_countries["topmalesellers"].alias("Countries_TopMaleSellers"),
    
    # Buyer stats
    silver_buyers["buyers"].alias("Buyers_Total"),
    silver_buyers["topbuyers"].alias("Buyers_Top"),
    silver_buyers["femalebuyers"].alias("Buyers_Female"),
    silver_buyers["malebuyers"].alias("Buyers_Male"),
    silver_buyers["topfemalebuyers"].alias("Buyers_TopFemale"),
    silver_buyers["topmalebuyers"].alias("Buyers_TopMale"),
    
    # Aggregated seller stats
    silver_sellers_agg["nbsellers"].alias("Sellers_Total"),
    silver_sellers_agg["meanproductssold"].alias("Sellers_MeanProductsSold"),
    silver_sellers_agg["meanproductslisted"].alias("Sellers_MeanProductsListed")
)

In [0]:
gold_base_path = "abfss://gold@ecomadlskyle.dfs.core.windows.net/"

root
 |-- identifierHash: string (nullable = true)
 |-- type: string (nullable = true)
 |-- country: string (nullable = true)
 |-- language: string (nullable = true)
 |-- socialnbfollowers: integer (nullable = true)
 |-- socialnbfollows: integer (nullable = true)
 |-- socialProductsLiked: string (nullable = true)
 |-- productsListed: string (nullable = true)
 |-- productsSold: string (nullable = true)
 |-- productspassrate: decimal(10,2) (nullable = true)
 |-- productsWished: string (nullable = true)
 |-- productsBought: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- civilityGenderId: string (nullable = true)
 |-- civilityTitle: string (nullable = true)
 |-- hasanyapp: boolean (nullable = true)
 |-- hasandroidapp: boolean (nullable = true)
 |-- hasiosapp: boolean (nullable = true)
 |-- hasprofilepicture: boolean (nullable = true)
 |-- dayssincelastlogin: integer (nullable = true)
 |-- seniority: string (nullable = true)
 |-- seniorityasmonths: decimal(10,2) (nullab

In [0]:
# Write the comprehensive user table to Gold layer
comprehensive_user_table.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_base_path}ecom_one_big_table")

In [0]:
# Move processed source files to archive folder
source_folder = "abfss://landing-zone-2@ecomadlskyle.dfs.core.windows.net/to_processed_user_data"
target_folder = "abfss://landing-zone-2@ecomadlskyle.dfs.core.windows.net/processed_user_data/user-raw-2/"

dbutils.fs.mkdirs(target_folder)

files_to_move = dbutils.fs.ls(source_folder)
for f in files_to_move:
    dbutils.fs.mv(f.path, target_folder + f.name)
    print(f"Moved: {f.name}")

In [0]:
# Preview the Gold table
df = spark.read.format("delta").load(f"{gold_base_path}ecom_one_big_table")
display(df.limit(20))

Country,Users_productsSold,Users_productsWished,Users_hasanyapp,Users_socialnbfollowers,Countries_Sellers,Countries_TopSellers,Countries_FemaleSellers,Countries_MaleSellers,Countries_TopFemaleSellers,Countries_TopMaleSellers,Buyers_Total,Buyers_Top,Buyers_Female,Buyers_Male,Buyers_TopFemale,Buyers_TopMale,Sellers_Total,Sellers_Sex,Sellers_MeanProductsSold,Sellers_MeanProductsListed
Andorre,0,0,false,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
Andorre,0,0,false,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
Andorre,0,0,true,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
Andorre,0,0,true,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
Andorre,0,0,false,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
Andorre,0,0,false,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
Andorre,0,0,true,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
Andorre,0,0,true,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
Andorre,0,0,false,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
Andorre,0,0,false,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [0]:
# =============================================================================
# DATA QUALITY REPORT
# =============================================================================
from pyspark.sql.functions import col, count, when, isnan, lit
from pyspark.sql.types import DoubleType, FloatType

df_check = spark.read.format("delta").load(f"{gold_base_path}ecom_one_big_table")
total_count = df_check.count()

# Check for duplicates
distinct_count = df_check.distinct().count()
duplicate_count = total_count - distinct_count

print(f"DATA QUALITY REPORT")
print(f"-" * 40)
print(f"Total rows: {total_count:,}")
print(f"Duplicate rows: {duplicate_count:,} ({(duplicate_count/total_count)*100:.2f}%)" if total_count > 0 else "Duplicate rows: 0")

# Check NULL rates per column
null_report = df_check.select([
    count(when(
        col(c).isNull() | (isnan(col(c)) if isinstance(df_check.schema[c].dataType, (DoubleType, FloatType)) else lit(False)),
        c
    )).alias(c)
    for c in df_check.columns
])

import pandas as pd
null_df = null_report.toPandas().transpose()
null_df.columns = ['null_count']
null_df['null_percentage'] = (null_df['null_count'] / total_count) * 100

print(f"\nNULL RATE BY COLUMN:")
display(null_df.sort_values(by='null_percentage', ascending=False))

📊 BÁO CÁO CHẤT LƯỢNG DỮ LIỆU
---------------------------------
✅ Tổng số dòng: 98913
🚫 Số dòng trùng lặp: 0 (0.00% )

🔍 CHI TIẾT TỈ LỆ NULL TRÊN TỪNG CỘT:


null_count,null_percentage
49222,49.7629229727134
49222,49.7629229727134
49222,49.7629229727134
49222,49.7629229727134
49222,49.7629229727134
49222,49.7629229727134
36864,37.2691152831276
36864,37.2691152831276
36864,37.2691152831276
35681,36.07311475741308


In [0]:
# ============================================================
# END-TO-END PIPELINE VALIDATION: BRONZE → SILVER → GOLD
# ============================================================
from pyspark.sql.functions import col

bronze_base_path = "abfss://bronze@ecomadlskyle.dfs.core.windows.net/"
silver_base_path_v = "abfss://silver@ecomadlskyle.dfs.core.windows.net/"
gold_base_path_v = "abfss://gold@ecomadlskyle.dfs.core.windows.net/"

# Read all layers
b_users = spark.read.format("delta").load(f"{bronze_base_path}users")
b_buyers = spark.read.format("delta").load(f"{bronze_base_path}buyers")
b_sellers = spark.read.format("delta").load(f"{bronze_base_path}sellers")
b_countries = spark.read.format("delta").load(f"{bronze_base_path}countries")

s_users = spark.read.format("delta").load(f"{silver_base_path_v}users")
s_buyers = spark.read.format("delta").load(f"{silver_base_path_v}buyers")
s_sellers = spark.read.format("delta").load(f"{silver_base_path_v}sellers")
s_countries = spark.read.format("delta").load(f"{silver_base_path_v}countries")

g_table = spark.read.format("delta").load(f"{gold_base_path_v}ecom_one_big_table")

print("=" * 65)
print("  END-TO-END PIPELINE VALIDATION REPORT")
print("=" * 65)

# ── 1. ROW COUNTS ACROSS LAYERS ────────────────────────────────
bc = {"users": b_users.count(), "buyers": b_buyers.count(),
      "sellers": b_sellers.count(), "countries": b_countries.count()}
sc = {"users": s_users.count(), "buyers": s_buyers.count(),
      "sellers": s_sellers.count(), "countries": s_countries.count()}
gc = g_table.count()

print("\n1. ROW COUNTS (Bronze → Silver → Gold)")
print(f"   {'Table':<12} {'Bronze':>10} {'Silver':>10} {'Change':>12}")
print(f"   {'-'*46}")
for t in ["users", "buyers", "sellers", "countries"]:
    diff = sc[t] - bc[t]
    sign = "+" if diff > 0 else ""
    print(f"   {t:<12} {bc[t]:>10,} {sc[t]:>10,} {sign + str(diff):>12}")
print(f"   {'-'*46}")
print(f"   {'Gold table':<12} {gc:>10,} rows")
print(f"   {'Expected':<12} {sc['users']:>10,} rows (= silver users)")
pass_gold_count = gc == sc["users"]
print(f"   Status: {'PASS' if pass_gold_count else 'FAIL'} — Gold rows {'match' if pass_gold_count else 'DO NOT match'} silver users")

# ── 2. DEDUPLICATION CHECK ─────────────────────────────────────
s_distinct = s_users.select("identifierHash").distinct().count()
g_distinct = g_table.select("IdentifierHash").distinct().count()
g_total_distinct = g_table.distinct().count()

print(f"\n2. DEDUPLICATION")
print(f"   Silver users distinct identifiers: {s_distinct:,}")
print(f"   Silver users total rows:           {sc['users']:,}")
pass_s_dedup = s_distinct == sc["users"]
print(f"   Silver dedup: {'PASS' if pass_s_dedup else 'FAIL'} — {'No' if pass_s_dedup else 'Has'} duplicates")
print(f"   Gold distinct identifiers:  {g_distinct:,}")
print(f"   Gold distinct rows (all cols): {g_total_distinct:,}")
pass_g_dedup = g_total_distinct == gc
print(f"   Gold dedup: {'PASS' if pass_g_dedup else 'FAIL'} — {'0' if pass_g_dedup else gc - g_total_distinct} duplicate rows")

# ── 3. SCHEMA EVOLUTION ────────────────────────────────────────
print(f"\n3. SCHEMA (column counts)")
print(f"   Bronze users:  {len(b_users.columns)} cols")
print(f"   Silver users:  {len(s_users.columns)} cols  (+{len(s_users.columns) - len(b_users.columns)} enriched)")
print(f"   Gold table:    {len(g_table.columns)} cols  (joined from 4 tables)")

# ── 4. JOIN INTEGRITY ──────────────────────────────────────────
users_countries = set(r[0] for r in s_users.select("country").distinct().collect() if r[0])
buyers_countries = set(r[0] for r in s_buyers.select("country").distinct().collect() if r[0])
sellers_countries = set(r[0] for r in s_sellers.select("country").distinct().collect() if r[0])
countries_countries = set(r[0] for r in s_countries.select("country").distinct().collect() if r[0])

matched_countries = users_countries & countries_countries
matched_buyers = users_countries & buyers_countries
matched_sellers = users_countries & sellers_countries

print(f"\n4. JOIN INTEGRITY")
print(f"   User countries: {len(users_countries)} distinct")
print(f"   Countries table: {len(countries_countries)} → {len(matched_countries)} matched ({len(matched_countries)/len(users_countries)*100:.0f}% coverage)")
print(f"   Buyers table:    {len(buyers_countries)} → {len(matched_buyers)} matched ({len(matched_buyers)/len(users_countries)*100:.0f}% coverage)")
print(f"   Sellers table:   {len(sellers_countries)} → {len(matched_sellers)} matched ({len(matched_sellers)/len(users_countries)*100:.0f}% coverage)")

# Verify no row explosion from sellers
print(f"   Sellers granularity: {s_sellers.count()} rows for {s_sellers.select('country').distinct().count()} countries")
print(f"   Sellers aggregated before join: PASS (Gold rows = Silver users)" if pass_gold_count else "   FAIL: Row explosion detected")

# ── 5. NULL ANALYSIS ───────────────────────────────────────────
print(f"\n5. NULL ANALYSIS (Gold table)")
non_null_users = g_table.filter(col("Users_productsSold").isNotNull()).count()
non_null_countries = g_table.filter(col("Countries_Sellers").isNotNull()).count()
non_null_buyers = g_table.filter(col("Buyers_Total").isNotNull()).count()
non_null_sellers = g_table.filter(col("Sellers_Total").isNotNull()).count()

print(f"   {'Column Group':<20} {'Non-Null':>10} {'Null':>10} {'Fill %':>8}")
print(f"   {'-'*50}")
for name, nn in [("User cols", non_null_users), ("Countries cols", non_null_countries),
                 ("Buyers cols", non_null_buyers), ("Sellers cols", non_null_sellers)]:
    null_ct = gc - nn
    pct = (nn / gc * 100) if gc > 0 else 0
    print(f"   {name:<20} {nn:>10,} {null_ct:>10,} {pct:>7.1f}%")

# ── 6. SILVER LAYER FIXES VERIFICATION ─────────────────────────
print(f"\n6. SILVER LAYER FIXES")

# Language mapping
lang_counts = s_users.groupBy("language_full").count().collect()
lang_dict = {r["language_full"]: r["count"] for r in lang_counts}
has_german = "German" in lang_dict and lang_dict["German"] > 0
all_other = len(lang_dict) == 1 and "Other" in lang_dict
print(f"   Language mapping: {'PASS' if has_german else 'FAIL'} — German: {lang_dict.get('German', 0):,}, "
      f"English: {lang_dict.get('English', 0):,}, French: {lang_dict.get('French', 0):,}, Other: {lang_dict.get('Other', 0):,}")

# Sellers None filter
none_sellers = s_sellers.filter(col("country").isin("None", "none", "NONE")).count()
print(f"   Sellers 'None' filter: {'PASS' if none_sellers == 0 else 'FAIL'} — {none_sellers} rows with 'None'")

# Countries division-by-zero
zero_sellers_ct = s_countries.filter((col("sellers") == 0) & col("top_seller_ratio").isNotNull()).count()
print(f"   Division-by-zero guard: {'PASS' if zero_sellers_ct == 0 else 'FAIL'}")

# Users overwrite mode (check no duplication)
print(f"   Users write mode (overwrite): {'PASS' if pass_s_dedup else 'FAIL'} — {sc['users']:,} rows, {s_distinct:,} distinct")

# ── 7. OVERALL VERDICT ─────────────────────────────────────────
all_pass = all([pass_gold_count, pass_s_dedup, pass_g_dedup, has_german, none_sellers == 0, zero_sellers_ct == 0])
print(f"\n{'='*65}")
if all_pass:
    print("  ✅ ALL CHECKS PASSED — Pipeline is healthy")
else:
    print("  ⚠️  SOME CHECKS FAILED — Review items above")
print(f"{'='*65}")

  END-TO-END PIPELINE VALIDATION REPORT

1. ROW COUNTS (Bronze → Silver → Gold)
   Table            Bronze     Silver       Change
   ----------------------------------------------
   users            98,913     98,913            0
   buyers               62         62            0
   sellers              73         70           -3
   countries            19         19            0
   ----------------------------------------------
   Gold table       98,913 rows
   Expected         98,913 rows (= silver users)
   Status: PASS — Gold rows match silver users

2. DEDUPLICATION
   Silver users distinct identifiers: 98,913
   Silver users total rows:           98,913
   Silver dedup: PASS — No duplicates
   Gold distinct identifiers:  98,913
   Gold distinct rows (all cols): 98,913
   Gold dedup: PASS — 0 duplicate rows

3. SCHEMA (column counts)
   Bronze users:  24 cols
   Silver users:  32 cols  (+8 enriched)
   Gold table:    21 cols  (joined from 4 tables)

4. JOIN INTEGRITY
   User co